In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, mean_absolute_error
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score, mean_absolute_error, accuracy_score
import joblib
import xgboost as xgb

# -------------------- Constants & Helper Functions -------------------- #
TEAM_MAPPING = {
    'Man Utd': 'Manchester Utd', 'Man United': 'Manchester Utd',
    'Man City': 'Manchester City', 'Newcastle Utd': 'Newcastle United',
    'Newcastle Ut': 'Newcastle United', "Nott'ham Forest": 'Nottingham Forest',
    'Paris S-G': 'Paris Saint-Germain', 'Inter Milan': 'Inter',
    'Spurs': 'Tottenham', 'West Ham Utd': 'West Ham United'
}

def clean_numeric_values(value):
    if isinstance(value, str):
        cleaned = value.replace(',', '').replace(' ', '')
        if cleaned.replace('.', '', 1).isdigit():
            return float(cleaned)
    return value

def standardize_team_names(df, column_name):
    df = df.copy()
    df.loc[:, column_name] = df[column_name].replace(TEAM_MAPPING).str.strip()
    return df

# -------------------- Data Loading & Preprocessing -------------------- #
def load_and_preprocess_data():
    stats_df = pd.read_csv('Combined_Leagues_Stats.csv').copy()
    fixtures_df = pd.read_csv('Fixture_Results.csv').copy()

    numeric_cols = [
        'progressive_carries', 'progressive_passes', 'xg', 'npxg',
        'xg_assist', 'npxg_xg_assist', 'goals_per90', 'assists_per90',
        'goals_assists_per90', 'goals_pens_per90', 'xg_per90',
        'xg_assist_per90', 'npxg_per90', 'Home_xG', 'Away_xG'
    ]

    for df in [stats_df, fixtures_df]:
        for col in numeric_cols:
            if col in df.columns:
                df.loc[:, col] = df[col].apply(clean_numeric_values)

    fixtures_df = fixtures_df.dropna(subset=['Home_xG', 'Away_xG']).copy()

    stats_df = standardize_team_names(stats_df, 'team')
    fixtures_df = standardize_team_names(fixtures_df, 'Home_Team')
    fixtures_df = standardize_team_names(fixtures_df, 'Away_Team')

    stats_df.loc[:, 'total_progression'] = (
        stats_df['progressive_carries'] + stats_df['progressive_passes']
    )

    feature_columns = [
        'total_progression', 'goals_per90', 'assists_per90',
        'goals_assists_per90', 'goals_pens_per90', 'xg_per90',
        'xg_assist_per90', 'npxg_per90'
    ]

    imputer = SimpleImputer(strategy='median')
    stats_df.loc[:, feature_columns] = imputer.fit_transform(stats_df[feature_columns])
    
    team_stats = stats_df.set_index('team')[feature_columns].to_dict('index')
    fixtures_df = fixtures_df.loc[fixtures_df['Home_Team'] != fixtures_df['Away_Team']].copy()
    
    return team_stats, fixtures_df, feature_columns

# -------------------- Training Execution -------------------- #
if __name__ == "__main__":
    team_stats, fixtures_df, feature_columns = load_and_preprocess_data()

    # Feature extraction
    def get_features(row):
        home_features = list(team_stats.get(row['Home_Team'].strip(), {}).values()) or [0]*len(feature_columns)
        away_features = list(team_stats.get(row['Away_Team'].strip(), {}).values()) or [0]*len(feature_columns)
        return home_features + away_features
    
    fixtures_df['features'] = fixtures_df.apply(get_features, axis=1)
    fixtures_df['result'] = fixtures_df.apply(
        lambda row: 2 if row['Home_Score'] > row['Away_Score'] else 1 if row['Home_Score'] == row['Away_Score'] else 0, 
        axis=1
    )

    X = np.array(fixtures_df['features'].tolist())
    y_class = fixtures_df['result'].values
    y_reg = fixtures_df[['Home_xG', 'Away_xG']].values

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_class_train, y_class_val = y_class[train_idx], y_class[val_idx]
        y_reg_train, y_reg_val = y_reg[train_idx], y_reg[val_idx]

        # Handle class imbalance
        smote = SMOTE(random_state=42)
        X_train_res, y_class_train_res = smote.fit_resample(X_train, y_class_train)
        y_reg_train_res = y_reg_train[np.arange(len(y_class_train_res)) % len(y_reg_train)]

        # Initialize models
        clf = xgb.XGBClassifier(
            objective='multi:softprob',
            num_class=3,
            eval_metric=['merror', 'mlogloss'],  # Track classification error
            early_stopping_rounds=10,
            random_state=42
        )
        
        reg_home = xgb.XGBRegressor(
            objective='reg:squarederror',
            eval_metric='mae',
            early_stopping_rounds=10,
            random_state=42
        )
        reg_away = xgb.XGBRegressor(
            objective='reg:squarederror',
            eval_metric='mae',
            early_stopping_rounds=10,
            random_state=42
        )

        # Train classifier with validation monitoring
        clf.fit(X_train_res, y_class_train_res, 
                eval_set=[(X_train_res, y_class_train_res), (X_val, y_class_val)],
                verbose=False)
        
        # Train regressors
        reg_home.fit(X_train_res, y_reg_train_res[:, 0], eval_set=[(X_val, y_reg_val[:, 0])], verbose=False)
        reg_away.fit(X_train_res, y_reg_train_res[:, 1], eval_set=[(X_val, y_reg_val[:, 1])], verbose=False)

        # Calculate accuracies
        train_class_pred = clf.predict(X_train_res)
        train_acc = accuracy_score(y_class_train_res, train_class_pred)
        
        val_class_pred = clf.predict(X_val)
        val_acc = accuracy_score(y_class_val, val_class_pred)
        val_f1 = f1_score(y_class_val, val_class_pred, average='macro')
        
        # Regression metrics
        val_home_xg = reg_home.predict(X_val)
        val_away_xg = reg_away.predict(X_val)
        val_mae = (mean_absolute_error(y_reg_val[:, 0], val_home_xg) + 
                   mean_absolute_error(y_reg_val[:, 1], val_away_xg)) / 2

        print(f"\nFold {fold+1} Metrics:")
        print(f"Train Accuracy: {train_acc:.4f}")
        print(f"Val Accuracy: {val_acc:.4f} | Val F1: {val_f1:.4f} | Val MAE: {val_mae:.4f}")
        print(f"Class Distribution - Train: {np.bincount(y_class_train_res)}, Val: {np.bincount(y_class_val)}")


        # Save models
        joblib.dump({
            'classifier': clf,
            'regressor_home': reg_home,
            'regressor_away': reg_away,
            'feature_columns': feature_columns,
            'team_stats': team_stats
        }, f'xgb_model_fold{fold+1}.pkl')

        # Evaluate
        val_class_pred = clf.predict(X_val)
        val_f1 = f1_score(y_class_val, val_class_pred, average='macro')
        val_home_xg = reg_home.predict(X_val)
        val_away_xg = reg_away.predict(X_val)
        val_mae = (mean_absolute_error(y_reg_val[:, 0], val_home_xg) + 
                   mean_absolute_error(y_reg_val[:, 1], val_away_xg)) / 2

        print(f"Fold {fold+1} Val F1: {val_f1:.4f} | Val MAE: {val_mae:.4f}")

    # Save artifacts
    joblib.dump({
        'team_stats': team_stats,
        'feature_columns': feature_columns
    }, 'xgb_artifacts.pkl')

C:\Users\DATA-JOHN\AppData\Local\Temp\ipykernel_10220\3749082487.py:68: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  stats_df.loc[:, feature_columns] = imputer.fit_transform(stats_df[feature_columns])



Fold 1 Metrics:
Train Accuracy: 0.7168
Val Accuracy: 0.4626 | Val F1: 0.4390 | Val MAE: 0.5691
Class Distribution - Train: [732 732 732], Val: [127 117 184]
Fold 1 Val F1: 0.4390 | Val MAE: 0.5691

Fold 2 Metrics:
Train Accuracy: 0.6894
Val Accuracy: 0.4673 | Val F1: 0.4588 | Val MAE: 0.5654
Class Distribution - Train: [733 733 733], Val: [127 118 183]
Fold 2 Val F1: 0.4588 | Val MAE: 0.5654

Fold 3 Metrics:
Train Accuracy: 0.7312
Val Accuracy: 0.4439 | Val F1: 0.4344 | Val MAE: 0.5632
Class Distribution - Train: [733 733 733], Val: [127 118 183]
Fold 3 Val F1: 0.4344 | Val MAE: 0.5632

Fold 4 Metrics:
Train Accuracy: 0.7322
Val Accuracy: 0.4790 | Val F1: 0.4624 | Val MAE: 0.5459
Class Distribution - Train: [733 733 733], Val: [127 118 183]
Fold 4 Val F1: 0.4624 | Val MAE: 0.5459

Fold 5 Metrics:
Train Accuracy: 0.6794
Val Accuracy: 0.4439 | Val F1: 0.4262 | Val MAE: 0.5366
Class Distribution - Train: [733 733 733], Val: [127 118 183]
Fold 5 Val F1: 0.4262 | Val MAE: 0.5366


In [3]:
# -------------------- Prediction Function -------------------- #
def predict_fixtures(fixtures_csv_path):
    artifacts = joblib.load('xgb_artifacts.pkl')
    team_stats = artifacts['team_stats']
    feature_columns = artifacts['feature_columns']
    
    # Load first fold's model
    model = joblib.load('xgb_model_fold1.pkl')
    clf = model['classifier']
    reg_home = model['regressor_home']
    reg_away = model['regressor_away']

    new_fixtures = pd.read_csv(fixtures_csv_path)
    new_fixtures = standardize_team_names(new_fixtures, 'Home_Team')
    new_fixtures = standardize_team_names(new_fixtures, 'Away_Team')

    # Feature extraction
    new_fixtures['features'] = new_fixtures.apply(
        lambda row: (
            list(team_stats.get(row['Home_Team'].strip(), {}).values()) +
            list(team_stats.get(row['Away_Team'].strip(), {}).values())
        ), axis=1
    )
    X_new = np.array(new_fixtures['features'].tolist())

    # Predict
    class_probs = clf.predict_proba(X_new)
    home_xg = reg_home.predict(X_new)
    away_xg = reg_away.predict(X_new)

    return pd.DataFrame({
        'Home_Team': new_fixtures['Home_Team'],
        'Away_Team': new_fixtures['Away_Team'],
        'Home_Win_Prob': class_probs[:, 2],
        'Draw_Prob': class_probs[:, 1],
        'Away_Win_Prob': class_probs[:, 0],
        'Predicted_Home_xG': home_xg,
        'Predicted_Away_xG': away_xg
    })

# Example usage:
# predictions = predict_fixtures('fixtures.csv')
# predictions['Prob_Diff'] = abs(predictions['Home_Win_Prob'] - predictions['Away_Win_Prob'])
# final_predictions = predictions.sort_values('Prob_Diff', ascending=False)

In [4]:
predictions = predict_fixtures('fixtures.csv')
predictions['Prob_Diff'] = abs(predictions['Home_Win_Prob'] - predictions['Away_Win_Prob'])
final_predictions = predictions.sort_values('Prob_Diff', ascending=False)

In [5]:
final_predictions.head()

,Home_Team,Away_Team,Home_Win_Prob,Draw_Prob,Away_Win_Prob,Predicted_Home_xG,Predicted_Away_xG,Prob_Diff
2,Manchester City,Leicester City,0.778469,0.107637,0.113893,2.417410,0.674507,0.664576
4,Southampton,Crystal Palace,0.166981,0.208464,0.624556,1.357772,1.861999,0.457575
5,Liverpool,Everton,0.597894,0.249449,0.152657,1.624592,0.840557,0.445236
0,Bournemouth,Ipswich Town,0.626696,0.176039,0.197265,1.787912,1.011014,0.429431
7,Atlético Madrid,Barcelona,0.117306,0.337194,0.545500,1.297578,1.841334,0.428194
